# Benchmarking Climate line plot

In [1]:
import sys, csv, json, gc
from pathlib import Path

sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

ROOT_DIR = next(p for p in [Path().resolve(), *Path().resolve().parents]
                if (p / "benchmarking").is_dir())
sys.path.insert(0, str(ROOT_DIR / "benchmarking"))

from utils.quanti_benchmarking_1_details import ask_question, ask_question_mistral, ask_question_gemma
from utils.quanti_benchmarking_1_analysis import normalize_number

CLIMATE_DIR = ROOT_DIR / "climate_pilot"
OUTPUT_DIR = Path().resolve() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# ground truth, keyed by image number
ground_truth = {row["num"]: row for row in csv.DictReader(open(CLIMATE_DIR / "ground_truth_climate.csv"))}
print(f"{len(ground_truth)} ground-truth rows loaded")



25 ground-truth rows loaded


In [2]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, Qwen3VLForConditionalGeneration

MODEL_REGISTRY = [
    {"label": "Gemma-12B",       "slug": "gemma4-12b",      "checkpoint": "google/gemma-4-12B-it",                        "loader": "gemma"},
    {"label": "Gemma-E4B",       "slug": "gemma4-e4b",      "checkpoint": "google/gemma-4-E4B-it",                        "loader": "gemma"},
    {"label": "Ministral-3-14B", "slug": "ministral-3-14b", "checkpoint": "mistralai/Ministral-3-14B-Instruct-2512-BF16", "loader": "mistral"},
    {"label": "Ministral-3-8B",  "slug": "ministral-3-8b",  "checkpoint": "mistralai/Ministral-3-8B-Instruct-2512-BF16",  "loader": "mistral"},
    {"label": "Qwen3-VL-4B",     "slug": "qwen3-vl-4b",     "checkpoint": "Qwen/Qwen3-VL-4B-Instruct",                    "loader": "qwen"},
    {"label": "Qwen3-VL-8B",     "slug": "qwen3-vl-8b",     "checkpoint": "Qwen/Qwen3-VL-8B-Instruct",                    "loader": "qwen"},
]

ASK_FN = {"gemma": ask_question_gemma, "mistral": ask_question_mistral, "qwen": ask_question}


def load_model(entry):
    if entry["loader"] == "qwen":
        processor = AutoProcessor.from_pretrained(entry["checkpoint"])
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            entry["checkpoint"], torch_dtype=torch.bfloat16, device_map="auto"
        ).eval()
    elif entry["loader"] == "mistral":
        processor = AutoProcessor.from_pretrained(entry["checkpoint"], fix_mistral_regex=True)
        model = AutoModelForImageTextToText.from_pretrained(
            entry["checkpoint"], torch_dtype=torch.bfloat16, device_map="auto"
        ).eval()
    else:  # gemma
        processor = AutoProcessor.from_pretrained(entry["checkpoint"])
        model = AutoModelForImageTextToText.from_pretrained(
            entry["checkpoint"], torch_dtype=torch.bfloat16, device_map="auto", attn_implementation="sdpa"
        ).eval()
    model.generation_config.max_length = None
    device = next(model.parameters()).device
    return model, processor, device


def unload_model(model, processor):
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

In [3]:
QUESTIONS = {
    "post_claim_correct": "Does the text in the post accurately describe the chart? Reply with only 'correct' or 'incorrect'.",
    "solar_year10": "What is the Solar investment value at Year 10? Reply with just the number.",
    "wind_year10": "What is the Wind investment value at Year 10? Reply with just the number.",
}

images = []
for p in sorted((CLIMATE_DIR / "posts/correct/PNGs").glob("*.png")):
    images.append((p.stem, str(p), "correct"))
for p in sorted((CLIMATE_DIR / "posts/incorrect/PNGs").glob("*.png")):
    images.append((p.stem, str(p), "incorrect"))

print(f"{len(images)} images ({sum(1 for _,_,v in images if v=='correct')} correct, "
      f"{sum(1 for _,_,v in images if v=='incorrect')} incorrect)")

50 images (25 correct, 25 incorrect)


In [4]:
for entry in MODEL_REGISTRY:
    out_path = OUTPUT_DIR / f"climate_benchmark_{entry['slug']}.json"
    if out_path.exists():
        print(f"⏭ {entry['label']}: already done, skipping")
        continue

    print(f"\n{'='*60}\nLoading {entry['label']} ({entry['checkpoint']})\n{'='*60}")
    model, processor, device = load_model(entry)
    ask_fn = ASK_FN[entry["loader"]]

    results = []
    for image_name, image_path, variant in images:
        num = image_name.split("_")[0]
        answers = {}
        for q_key, q_text in QUESTIONS.items():
            answers[q_key] = ask_fn(image_path, q_text, model, processor, device)
        results.append({"image": image_name, "num": num, "variant": variant, "answers": answers})
        print(f"  {image_name}: {answers}")

    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"✅ Saved: {out_path}")

    unload_model(model, processor)

⏭ Gemma-12B: already done, skipping
⏭ Gemma-E4B: already done, skipping
⏭ Ministral-3-14B: already done, skipping
⏭ Ministral-3-8B: already done, skipping

Loading Qwen3-VL-4B (Qwen/Qwen3-VL-4B-Instruct)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

  001_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '54.6', 'wind_year10': '47.3'}
  002_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '40.9', 'wind_year10': '36.2'}
  003_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '30.4', 'wind_year10': '25.9'}
  004_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '39.7', 'wind_year10': '34.2'}
  005_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '31.2', 'wind_year10': '24.5'}
  006_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_year10': '12.6', 'wind_year10': '7.1'}
  007_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '28.4', 'wind_year10': '20.9'}
  008_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '20.9', 'wind_year10': '14.8'}
  009_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '42.5', 'wind_year10': '36.8'}
  010_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_year10'

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

  001_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '54.6', 'wind_year10': '47.3'}
  002_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '40.9', 'wind_year10': '36.2'}
  003_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '32.4', 'wind_year10': '25.9'}
  004_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '39.7', 'wind_year10': '34.2'}
  005_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '31.2', 'wind_year10': '24.5'}
  006_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_year10': '12.6', 'wind_year10': '7.1'}
  007_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_year10': '28.4', 'wind_year10': '20.9'}
  008_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_year10': '20.9', 'wind_year10': '14.8'}
  009_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '42.5', 'wind_year10': '36.8'}
  010_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_yea

In [7]:
def score_model(slug):
    path = OUTPUT_DIR / f"climate_benchmark_{slug}.json"
    if not path.exists():
        return None
    results = json.loads(path.read_text())
    scores = {"post_claim_correct": [], "solar_year10": [], "wind_year10": []}
    for r in results:
        gt = ground_truth[r["num"]]
        pred_claim = r["answers"]["post_claim_correct"].strip().lower()
        scores["post_claim_correct"].append(pred_claim == r["variant"])

        pred_solar = normalize_number(r["answers"]["solar_year10"])
        true_solar = normalize_number(gt["solar_final_year"])
        scores["solar_year10"].append(pred_solar == true_solar)

        pred_wind = normalize_number(r["answers"]["wind_year10"])
        true_wind = normalize_number(gt["wind_final_year"])
        scores["wind_year10"].append(pred_wind == true_wind)
    return {q: 100 * sum(v) / len(v) for q, v in scores.items()}


print(f"{'model':17s}{'post_claim_correct':>20s}{'solar_year10':>16s}{'wind_year10':>14s}")
for entry in MODEL_REGISTRY:
    acc = score_model(entry["slug"])
    if acc is None:
        print(f"{entry['label']:17s}  no data")
        continue
    print(f"{entry['label']:17s}{acc['post_claim_correct']:20.2f}{acc['solar_year10']:16.2f}{acc['wind_year10']:14.2f}")

model              post_claim_correct    solar_year10   wind_year10
Gemma-12B                       94.00           76.00         88.00
Gemma-E4B                       50.00           78.00         74.00
Ministral-3-14B                 50.00           96.00        100.00
Ministral-3-8B                  72.00           96.00         96.00
Qwen3-VL-4B                     58.00           92.00        100.00
Qwen3-VL-8B                     80.00           96.00         96.00
